### Question 3

In [52]:
#Import modules
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import train_test_split, HalvingGridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.metrics import roc_auc_score
import lightgbm
from xgboost import XGBClassifier
from sklearn.decomposition import PCA

In [2]:
#import datasets
train_dataset = pd.read_csv(".\\Data\\train.csv", header=0)
test_dataset = pd.read_csv(".\\Data\\test.csv", header=0)

### Exploratory Data Analysis

In [3]:
#copy new dataframe for data exploration and manipulation
new_train_df = train_dataset.copy(deep=True)

In [4]:
#check the values in dataframe are organised
new_train_df.head()

,created_at,default_profile,default_profile_image,description,favourites_count,followers_count,friends_count,geo_enabled,id,lang,location,profile_background_image_url,profile_image_url,screen_name,statuses_count,verified,average_tweets_per_day,account_age_days,target
0,2012-01-15 23:40:09,True,False,Cosplayer/Fitness lover. Come to me https://t....,74,7,0,False,465096524,en,unknown,http://abs.twimg.com/images/themes/theme1/bg.png,http://pbs.twimg.com/profile_images/9666745212...,reml5477,20,False,0.006,3138,1
1,2016-10-04 00:44:39,False,False,pobody’s nerfect,50443,164,590,True,783105517673648132,cy,she/her,http://abs.twimg.com/images/themes/theme1/bg.png,http://pbs.twimg.com/profile_images/1281752126...,kinlibra,6469,False,4.572,1415,0
2,2009-05-23 04:04:13,False,False,gracias por participar 🏅,9394,208,189,False,41970759,es,La diaspora,http://abs.twimg.com/images/themes/theme17/bg.gif,http://pbs.twimg.com/profile_images/1233811596...,_delaualau,30296,False,7.378,4106,0
3,2009-05-17 04:31:31,False,False,Stand Up Comedian/Actor from North Philadelphi...,46,66180,1090,True,40607946,en,"Calabasas, CA",http://abs.twimg.com/images/themes/theme1/bg.png,http://pbs.twimg.com/profile_images/1184851104...,SpankHorton,164957,False,40.116,4112,0
4,2009-02-16 13:11:21,True,False,Assignment Editor at NBC10 and President of Ja...,1223,487,867,True,20983433,en,"Jenkintown, PA",http://abs.twimg.com/images/themes/theme1/bg.png,http://pbs.twimg.com/profile_images/5234863934...,javelinjt,1752,False,0.417,4201,0


In [5]:
#check the datatypes of values in each column
new_train_df.dtypes

created_at                       object
default_profile                    bool
default_profile_image              bool
description                      object
favourites_count                  int64
followers_count                   int64
friends_count                     int64
geo_enabled                        bool
id                                int64
lang                             object
location                         object
profile_background_image_url     object
profile_image_url                object
screen_name                      object
statuses_count                    int64
verified                           bool
average_tweets_per_day          float64
account_age_days                  int64
target                            int64
dtype: object

In [6]:
#Check for missing values in dataset
new_train_df.isna().sum()

created_at                         0
default_profile                    0
default_profile_image              0
description                     5091
favourites_count                   0
followers_count                    0
friends_count                      0
geo_enabled                        0
id                                 0
lang                            5588
location                           2
profile_background_image_url    3235
profile_image_url                  1
screen_name                        0
statuses_count                     0
verified                           0
average_tweets_per_day             0
account_age_days                   0
target                             0
dtype: int64

Only columns with categorical variables have missing values so only need to impute for categorical variables

In [7]:
#Create placeholder to hold for newly_created columns - for processing of test dataset
new_cols = []

In [8]:
#Create function to process dataframe to required format
def df_feature_engineering(df):
    if "created_at" in df.columns: #bypass df if it has been processed before
        #Feature engineer new columns from existing columns
        df["favorites_status_ratio"] = df["favourites_count"]/(df["statuses_count"]+1) #check for engagement imbalance between status_count and favorites_count to be used (assume bots have more tweets than favorites); +1 to denominator (smoothing) to prevent zero-divisor error
        df["followers_friends_ratio"] = df["followers_count"]/(df["friends_count"]+1) #bots tend to have more followings than followers; +1 to denominator (smoothing) to prevent zero-divisor error
        df["status_counts_per_follower"] = df["statuses_count"]/(df["followers_count"]+1) #check if there is a excessive tweets against followers count
        keywords_to_find = ["bot","http"]
        for keyword in keywords_to_find:
            df[keyword+"_in_description"] = df["description"].str.contains(r"{}".format(keyword), case=False, na=False) #search for bot-like description
        df["length_of_description"] = df["description"].fillna("").str.len() #bots more likely to have shorter descriptions
        #Drop non-predictive columns
        df.drop(columns=["created_at","description","id","location","profile_background_image_url","profile_image_url","screen_name"], inplace=True)
        # Create indicator columns
        if "target" in df.columns: #for training data
            for col_name in df.columns.drop("target"):
                if df[col_name].isna().any(): #create missing values indicator columns
                    df[col_name+"_missing"] = df[col_name].isna()
                    df[col_name] = df[col_name].fillna("missing")
                    new_cols.append(col_name + "_missing")
        else: #for test dataset
            for col_name in new_cols:
                if col_name not in df.columns:
                    col = col_name.replace("_missing","")
                    df[col_name] = df[col].isna()      
                    df[col] = df[col].fillna("missing")
                    
            if "index" in df.columns:
                df.drop(columns=["index"], inplace=True)
    return df

Hour is extracted from the "created_at" column in the dataset as I believe that it might be correlated to the target variable.

In [9]:
df_feature_engineering(new_train_df) #Process dataset (feature engineer)

,default_profile,default_profile_image,favourites_count,followers_count,friends_count,geo_enabled,lang,statuses_count,verified,average_tweets_per_day,account_age_days,target,favorites_status_ratio,followers_friends_ratio,status_counts_per_follower,bot_in_description,http_in_description,length_of_description,lang_missing
0,True,False,74,7,0,False,en,20,False,0.006,3138,1,3.523810,7.000000,2.500000,False,True,59,False
1,False,False,50443,164,590,True,cy,6469,False,4.572,1415,0,7.796445,0.277496,39.206061,False,False,16,False
2,False,False,9394,208,189,False,es,30296,False,7.378,4106,0,0.310064,1.094737,144.956938,False,False,24,False
3,False,False,46,66180,1090,True,en,164957,False,40.116,4112,0,0.000279,60.659945,2.492513,False,False,147,False
4,True,False,1223,487,867,True,en,1752,False,0.417,4201,0,0.697661,0.561060,3.590164,False,False,55,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26201,True,False,72150,358,417,True,missing,9882,False,3.629,2723,0,7.300415,0.856459,27.526462,False,False,0,True
26202,True,False,13603,294,208,True,missing,4347,False,1.905,2282,0,3.128565,1.406699,14.735593,False,False,0,True
26203,False,False,2300,1132,819,True,sw,51607,False,13.982,3691,0,0.044567,1.380488,45.548985,False,False,3,False
26204,False,False,131578,457,358,True,en,23232,False,5.633,4124,0,5.663410,1.272981,50.724891,False,False,106,False


In [10]:
#add column names to be passed into preprocessing modules so that they know which columns to be processed
categorical_col = list(new_train_df.select_dtypes(include=["object"]).columns) #create list of categorical column names to input into OneHotEncoder
numeric_col = list(new_train_df.select_dtypes(include=["bool","int64","int32","float64"]).columns.drop('target')) #create list of categorical column names to input into OneHotEncoder

In [69]:
#add collinearity split - lgmboost
#Preprocess dataset and set up stacking pipeline - hyperparameters set for models were discovered using hyperparameter tuning techniques previously
preprocessor = ColumnTransformer([("num_transformer",StandardScaler(),numeric_col),("cat_transformer", OneHotEncoder(handle_unknown="ignore", sparse_output=False),categorical_col)], remainder="passthrough")
estimators = [('rcf_model',RandomForestClassifier(n_estimators=100, min_samples_leaf = 5, max_features = 0.5, max_depth = 20, random_state=2025)),("lgm_boost",lightgbm.LGBMClassifier(metric="auc", feature_fraction=0.5, learning_rate=0.1, n_estimators=200, max_depth=10, random_state=2025)),("knn", KNeighborsClassifier(n_neighbors=5,weights='distance',metric="manhattan"))]
meta_pipeline = Pipeline([('pca',PCA(n_components=2)),('meta_estimator',LogisticRegression(C=10, random_state=2025))])
stacked_models = StackingClassifier(estimators=estimators, final_estimator=meta_pipeline, cv=5, passthrough=False, verbose=1) #Perform stacking of models; passthrough to make sure the subsequent models will only see the features passed down, not the original set
stacked_pipeline = Pipeline([("preprocess", preprocessor),("stacking", stacked_models)])
#Split dataset into X and Y 
X = new_train_df.drop(columns="target")
y = new_train_df["target"]
#split dataset into training and test dataset and fit into model via cross_val_score (cross validation approach)
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=2025, stratify=y)
stacked_pipeline.fit(X_train,y_train)
y_pred_proba = stacked_pipeline.predict_proba(X_test)
auc_score = roc_auc_score(y_test, y_pred_proba[:,1])
print("AUC Score:",auc_score)

[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Info] Number of positive: 7026, number of negative: 13938
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002141 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2555
[LightGBM] [Info] Number of data points in the train set: 20964, number of used features: 49
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.335146 -> initscore=-0.685001
[LightGBM] [Info] Start training from score -0.685001
[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] 

In [70]:
# Process test dataset to same data structure as training dataset to fit into the model for prediction
new_test_df = test_dataset.copy(deep=True)
df_feature_engineering(new_test_df)

,default_profile,default_profile_image,favourites_count,followers_count,friends_count,geo_enabled,lang,statuses_count,verified,average_tweets_per_day,account_age_days,favorites_status_ratio,followers_friends_ratio,status_counts_per_follower,bot_in_description,http_in_description,length_of_description,lang_missing
0,False,False,22290,163813,1746,True,en,42013,True,8.586,4893,0.530537,93.768174,0.256468,False,False,154,False
1,True,False,685,44,637,False,missing,111,False,0.041,2699,6.116071,0.068966,2.466667,False,False,0,True
2,False,False,12257,276,194,True,de,18142,False,4.390,4133,0.675577,1.415385,65.494585,False,False,16,False
3,True,False,30806,1124,4999,True,en,20624,False,5.627,3665,1.493624,0.224800,18.332444,False,False,77,False
4,True,False,224,22,265,False,en,64,False,0.032,1996,3.446154,0.082707,2.782609,False,True,59,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11227,False,False,16886,1973,238,True,tl,132560,False,36.318,3650,0.127383,8.255230,67.152989,False,False,3,False
11228,True,False,0,4263,0,False,en,1,False,0.001,1059,0.000000,4263.000000,0.000235,False,False,111,False
11229,False,False,1530,367,245,False,en,11355,False,2.896,3921,0.134731,1.491870,30.855978,False,False,22,False
11230,False,False,1460,425666,540,False,en,2549,True,0.626,4072,0.572549,786.813309,0.005988,False,False,20,False


In [71]:
#Predict using test_dataset using new RandomForest Model
y_test_pred = stacked_pipeline.predict_proba(new_test_df)

[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5


In [72]:
#Export prediction probabilities for submission
test_pred = pd.DataFrame({"index": np.arange(len(new_test_df)), "target":y_test_pred[:,1]})
test_pred.to_csv("01521730_A1Q3_submission.csv", index=False)

### <u> Concluding Section for Question 3 </u>

Approach taken: <br>
<b>(1) Feature Engineering<br></b>
- Created ratios for analysis
- Added in len_of_description as bots tend to have shorter description
- Added in columns to indicate presence of bot-like words
- Added in indicator columns to indicate which feature has missing values
- Remove non-predictive columns

<br>
<b>(2) Stacking <br></b>
Base Estimators used: (1) RandomForestClassifier, (2) LGMBoost Model, (3) KNNClassifier Model <br>
Meta-Estimator: Logistic Regression Model <br>

Kaggle Score: 0.93955